# 의사결정2 3주차 — My Business × AI Harness Reconstruction

## 학습 목표

기존 강의의 Operations Research 문제를
내 실제 사업과 AI Harness 운영문제로 다시 모델링한다.

각 개념은 두 번 푼다.

### A. Human / Principal Model

인간 류지환의 관점에서

- 어떤 상품을 몇 건 받을 것인가?
- 수익 목표를 달성할 수 있는가?
- 인간시간은 충분한가?
- AI budget은 충분한가?

를 결정한다.

### B. Harness / Orchestrator Model

이미 프로젝트가 수주된 뒤 Top Orchestrator의 관점에서

- 어떤 Harness 구조를 선택할 것인가?
- Closure deadline을 만족하는가?
- Human Review를 얼마나 요구하는가?
- AI token을 얼마나 사용하는가?
- Independent Verification을 확보할 수 있는가?

를 결정한다.

---

# 전체 난이도

## Level 1
Feasibility

"모든 목표를 동시에 만족할 수 있는가?"

## Level 2
Goal Programming

"전부 만족할 수 없다면 무엇부터 지킬 것인가?"

## Level 3
Resource Sensitivity

"인간시간 또는 Token을 줄이면 결과가 어떻게 변하는가?"

## Level 4
Pareto / Multi-objective

"두 목표 사이의 효율적 trade-off는 어디인가?"

## Level 5
Detailed Integer Programming

"상품 또는 Agent를 개별 decision variable로 직접 고른다."

## Level 6
Dynamic Resource Expansion

"추가 인간시간 / 추가 Agent / 추가 Token이 언제 가치가 있는가?"

## Final

백지에서 Human Model과 Harness Model을 직접 재구성한다.

In [1]:
import pandas as pd
from ortools.linear_solver import pywraplp
from IPython.display import display, Markdown

def md(title, body):
    """
    계산 결과를 Markdown 형식으로 출력하는 보조함수.

    title : 해석 블록의 제목
    body  : 분석 결과에 대한 자연어 설명
    """
    
    display(
        Markdown(
            f"### {title}\n\n{body}"
        )
    )

STATUS_LABEL = {
    pywraplp.Solver.OPTIMAL: "OPTIMAL",
    pywraplp.Solver.FEASIBLE: "FEASIBLE",
    pywraplp.Solver.INFEASIBLE: "INFEASIBLE",
    pywraplp.Solver.UNBOUNDED: "UNBOUNDED",
    pywraplp.Solver.NOT_SOLVED: "NOT_SOLVED",
}


md(
    "환경 설정 완료",
    "이제부터 모든 결과를 **수식 → Solver → DataFrame → 자연어 해석** 순서로 확인한다."
)


### 환경 설정 완료

이제부터 모든 결과를 **수식 → Solver → DataFrame → 자연어 해석** 순서로 확인한다.

# Level 1-A — Human 류지환 Feasibility Problem

류지환은 다음 네 종류의 MVP 프로젝트를 판매한다.

| 상품 | 의미 | 공헌이익 | Human Time | AI Token |
|---|---|---:|---:|---:|
| P0 | Product MVP | 20만원 | 6h | 60,000 |
| P1 | Build MVP | 30만원 | 12h | 140,000 |
| P2 | Data MVP | 40만원 | 14h | 180,000 |
| P3 | Full Package | 100만원 | 28h | 420,000 |

AI token은 계산을 편하게 하기 위해

$$
1U=10,000\text{ tokens}
$$

로 정의한다.

따라서 각 상품의 token 사용량은

- P0 = 6U
- P1 = 14U
- P2 = 18U
- P3 = 42U

이다.

---

## 이번 달 희망목표

### Goal 1 — 공헌이익

적어도 210만원을 벌고 싶다.

$$
20x_0+30x_1+40x_2+100x_3\ge210
$$

### Goal 2 — Human Time

프로젝트 생산에 직접 사용하는 시간은
60시간 이하이고 싶다.

$$
6x_0+12x_1+14x_2+28x_3\le60
$$

### Goal 3 — AI Token

AI 사용량은 650,000 tokens 이하이고 싶다.

$$
6x_0+14x_1+18x_2+42x_3\le65
$$

### Goal 4 — Full Package

Full Package를 적어도 한 건은 수행하고 싶다.

$$
x_3\ge1
$$

---

## 질문

현재 단계에서는 무엇을 최대화하지 않는다.

단지

> 네 목표를 전부 동시에 만족하는 사업계획이 존재하는가?

만 확인한다.

따라서 목적함수는

$$
\boxed{\min 0}
$$

이다.

In [2]:


human_product_data = pd.DataFrame({
    
    "product": [
        "P0",
        "P1",
        "P2",
        "P3",
    ],
    # product:
    # 의사결정 대상이 되는 상품의 ID이다.
    
    
    "service": [
        "Product MVP",
        "Build MVP",
        "Data MVP",
        "Full Package",
    ],
    # service:
    # 읽기 위한 상품 이름이다.
    
    
    "profit_만원": [
        20,
        30,
        40,
        100,
    ],
    # profit_만원:
    # 프로젝트 한 건을 수행할 때 얻는 이익이다.
    
    
    "human_hours": [
        6,
        12,
        14,
        28,
    ],
    # human_hours:
    # 프로젝트 한 건당 인간 류지환의 직접 생산시간(인시)이다.
    
    
    "token_U": [
        6,
        14,
        18,
        42,
    ],
    # token_U:
    # 프로젝트 한 건당 AI 사용량이다.
    #
    # 여기서 1U = 10,000 tokens.
})


# 실제 token 수도 별도 열로 만든다.
human_product_data["tokens"] = (
    human_product_data["token_U"] * 10_000
)


# 표를 출력한다.
display(human_product_data)

,product,service,profit_만원,human_hours,token_U,tokens
0,P0,Product MVP,20,6,6,60000
1,P1,Build MVP,30,12,14,140000
2,P2,Data MVP,40,14,18,180000
3,P3,Full Package,100,28,42,420000


$$14U=140,000 tokens/project$$

# Human Model의 Decision Variable

이번 달에 수행할 프로젝트 수를 결정한다.


$$
x_0=\text{P0 Product MVP 수}
$$

$$
x_1=\text{P1 Build MVP 수}
$$

$$
x_2=\text{P2 Data MVP 수}
$$

$$
x_3=\text{P3 Full Package 수}
$$

프로젝트 수는

- 음수가 될 수 없고
- 0.4건 같은 값도 현실적으로 불가능하다.

따라서

$$
\boxed{
x_0,x_1,x_2,x_3
\in\mathbb Z_{\ge0}
}
$$

이다.

즉 이번 문제는 연속형 LP가 아니라
정수계획 문제이다.

In [3]:
# Human Feasibility Model — Solver와 Decision Variable 생성


solver_human = pywraplp.Solver.CreateSolver("SCIP")
# SCIP = 정수형 의사결정 변수를 포함하는 문제를 풀 수 있는 -- MUILTI SOLVER

if solver_human is None:
    raise RuntimeError("SCIP solver를 실행하지 못함.")
# 솔버 자체가 생성 실패한경우.
# 솔버 자체가 생성 실패할시 결과를 생성하지 않도록함. -> 오류발생

x0 = solver_human.IntVar(
    0,
    solver_human.infinity(),
    "P0_Product_MVP",
)
#이때 의사결정변수 X_0는 이번달 프로덕트 MVP 수주 수에 대한 의사결정변수임.
#Lower_band = 0으롯 설정, 
#Upper_Band = Infinity.
#이때 type은 Int

x1 = solver_human.IntVar(
    0, 
    solver_human.infinity(),
    "P1_Build_MVP",
)
#이때의 의사결정 변수 X_1은 이번달 수주받을 Build MVP에 해당하는 프로젝트수 

x2 = solver_human.IntVar(
    0,
    solver_human.infinity(),
    "P2_Data_MVP",
)
#이때의 의사결정 변수 X_2는 이번달 수주받을 Data_MVP에 해당하는 프로젝트수 

x3 = solver_human.IntVar(
    0,
    solver_human.infinity(),
    "P3_Full_Package",
)
#이때의 의사결정 변수 X_3는 이번달 수주받을 P3_Full_Package에 해당하는 프로젝트수 

#------------
print(md("의사결정 변수 $$X_0 = P0$$", ""), x0)

print(md("의사결정 변수 $$X_1 = P1BuildMVP$$", ""), x1)

print(md("의사결정 변수 $$X_2 = P2DataMVP$$", ""), x2)

print(md("의사결정 변수 $$X_3 = P3FullPackage$$", ""), x3)




### 의사결정 변수 $$X_0 = P0$$



None P0_Product_MVP


### 의사결정 변수 $$X_1 = P1BuildMVP$$



None P1_Build_MVP


### 의사결정 변수 $$X_2 = P2DataMVP$$



None P2_Data_MVP


### 의사결정 변수 $$X_3 = P3FullPackage$$



None P3_Full_Package


# 제약조건을 하나의 표현으로 다시 정리

Decision Vector를

$$
x=
\begin{bmatrix}
x_0\\
x_1\\
x_2\\
x_3
\end{bmatrix}
$$

라고 하자.

그러면 이익은

$$
R(x)
=
20x_0+30x_1+40x_2+100x_3
$$

Human Time은

$$
H(x)
=
6x_0+12x_1+14x_2+28x_3
$$

AI Token은

$$
T(x)
=
6x_0+14x_1+18x_2+42x_3
$$

이다.

즉 같은 Decision Vector x에
서로 다른 coefficient vector를 곱하고 있다.

이번 Feasibility Problem의 조건은

$$
R(x)\ge210
$$

$$
H(x)\le60
$$

$$
T(x)\le65
$$

$$
x_3\ge1
$$

이다.

--

즉, 같은 

$$
x
$$

에 대해서

$$
x=
\begin{bmatrix}
x_0\\
x_1\\
x_2\\
x_3
\end{bmatrix}
$$

과

$$
R(x)=
\begin{bmatrix}
20\\
30\\
40\\
100\\
\end{bmatrix}
=
\begin{bmatrix}
x_0\\
x_1\\
x_2\\
x_3
\end{bmatrix}
$$

을 내적시에는 *이익*,  

$$
H(x)=
\begin{bmatrix}
6\\
12\\
14\\
28\\
\end{bmatrix}
=
\begin{bmatrix}
x_0\\
x_1\\
x_2\\
x_3
\end{bmatrix}
$$

과 내적하면 *인시*

-> 즉, 서로 다른 **mesurement Axis**로 설정한다

In [4]:
# Human Feasibility Model — 제약조건 정의

profit_expr = (
    20 * x0
    + 30 * x1
    + 40 * x2
    + 100 * x3
)
print(md("이때 profit_expr는,$$ProfitExpr = 20 * x_0 + 30 * x_1 + 40 * x_2 + 100 * x_3$$", "---")) 

human_expr = (
    6 * x0
    + 12 * x1
    + 14 * x2
    + 28 * x3
)
print(md("이때 human_expr는,$$HumanExpr = 6 * x_0 + 12 * x_1 + 14 * x_2 + 28 * x_3$$", "---")) 

token_expr = (
    6 * x0
    + 14 * x1
    + 18 * x2
    + 42 * x3
)
print(md("이때 *token_expr* 는, 총 AI 사용량이며 단위는 $$1U = 10,000 tokens$$ $$TokenExpr = 6 * x_0 + 14 * x_1 + 18 * x_2 + 42 * x_3$$", "---"))
#

solver_human.Add(
    profit_expr >= 210,
    "Goal1_Profit_AtLeast_210",
)
print(md("이때 *profit_expr* 는, 하한조건이며, $$ ProfitExpr >= 210 Won$$ 이익은 최소 210만원 이상.", "---")) 


solver_human.Add(
    human_expr <= 60,
    "Goal2_Human_AtMost_60",
)
print(md("이때 *human_expr* 는, 상한조건이며, $$ HumanExpr <= 60 hours$$ 류지환의 시간을 달 기준 최대 60시간 이상을 뺐지 마라", "---")) 



solver_human.Add(
    token_expr <= 65,
    "Goal3_Token_AtMost_650K",
)
print(md("이때 *token_expr* 는, 상한조건이며, $$ TokenExpr <= 65*U$$ 즉 650,000token 이하로 사용하라. 류지환은 돈이없다", "---")) 

#
solver_human.Add(
    x3 >= 1,
    "Goal4_FullPackage_AtLeast_by1",
)
print(md("이때 *FullPackage* 는, 하한조건이며, $$ x3 >= 1 $$ 류지환은 경험이 고프다", "---"))



### 이때 profit_expr는,$$ProfitExpr = 20 * x_0 + 30 * x_1 + 40 * x_2 + 100 * x_3$$

---

None


### 이때 human_expr는,$$HumanExpr = 6 * x_0 + 12 * x_1 + 14 * x_2 + 28 * x_3$$

---

None


### 이때 *token_expr* 는, 총 AI 사용량이며 단위는 $$1U = 10,000 tokens$$ $$TokenExpr = 6 * x_0 + 14 * x_1 + 18 * x_2 + 42 * x_3$$

---

...

None


### 이때 *token_expr* 는, 상한조건이며, $$ TokenExpr <= 65*U$$ 즉 650,000token 이하로 사용하라. 류지환은 돈이없다

---

None


### 이때 *FullPackage* 는, 하한조건이며, $$ x3 >= 1 $$ 류지환은 경험이 고프다

---

None


In [5]:

##
solver_human.Minimize(0)

print(md("이때 *solver_human.Minimize(0)* 는, 중요하다. 왜? 최적화 solver를 사용하나, 지금은 **Constrataint_Satisfaction**체크가 우선이므로. $$ Step_1 = Minimize(0) $$", "---"))

## step1 결과 : 
status_human = solver_human.Solve()

print(status_human, md(" **STEP1** 지금까지 선언한 $$decision variable + constraints + objective$$를 풀도록", " ### 숫자로 반환되는 solver 상태를 사람이 읽을 수 있는 문자열로 변환한다."))

status_human_text = STATUS_LABEL.get(
    status_human,
    f"STATUS_{status_human}",
)

print("Solver_Status =", status_human_text)

print("STEP1 Human Feasibility 결과를 DataFrame으로 구조화하면,")

#dataframe

human_goal_table = pd.DataFrame([
    {
        "목표": "Profit이익",
        "direction": ">=",
        "조건": 210,
        "단위": "만원",
        "의미": "월 최소 210먼원 이상",
    },

    {
        "목표": "HumanTime_인시",
        "direction": "<=",
        "조건": 60,
        "단위": "시간",
        "의미": "60시간 이하로 투입",
    },

    {
        "목표": "AI Token_토큰사용량",
        "direction": "<=",
        "조건": 650_000,
        "단위": "tokens",
        "의미": "토큰사용량 65만토큰 이하",
    },

    {
        "목표": "Full_package",
        "direction": ">=",
        "조건": 1,
        "단위": "project",
        "의미": "P3최소 1건 이상",
    },
])

# 모든 목표를 묶은 문제의 Solver 상태를 붙인다.
human_goal_table["STEP1현황"] = status_human_text

display(human_goal_table)

print("STEP1 최종결과")
print(f"4 목표를 동시에 강제한 결과 = {status_human_text}")
print("현재 어떤 조합도 네개의 목표를 전부 동시에 만족시키지 못한다")

### 이때 *solver_human.Minimize(0)* 는, 중요하다. 왜? 최적화 solver를 사용하나, 지금은 **Constrataint_Satisfaction**체크가 우선이므로. $$ Step_1 = Minimize(0) $$

---

None


###  **STEP1** 지금까지 선언한 $$decision variable + constraints + objective$$를 풀도록

 ### 숫자로 반환되는 solver 상태를 사람이 읽을 수 있는 문자열로 변환한다.

2 None
Solver_Status = INFEASIBLE
STEP1 Human Feasibility 결과를 DataFrame으로 구조화하면,


,목표,direction,조건,단위,의미,STEP1현황
0,Profit이익,>=,210,만원,월 최소 210먼원 이상,INFEASIBLE
1,HumanTime_인시,<=,60,시간,60시간 이하로 투입,INFEASIBLE
2,AI Token_토큰사용량,<=,650000,tokens,토큰사용량 65만토큰 이하,INFEASIBLE
3,Full_package,>=,1,project,P3최소 1건 이상,INFEASIBLE


STEP1 최종결과
4 목표를 동시에 강제한 결과 = INFEASIBLE
현재 어떤 조합도 네개의 목표를 전부 동시에 만족시키지 못한다


# STEP1 결과해석 : 왜 불가능한지 최소한만 진단

다음 Goal Programming으로 넘어가기 위해서.

In [24]:
# Human Feasibility — Profit 목표가 얼마나 과한지 진단

diagnose_human = pywraplp.Solver.CreateSolver("SCIP")
# 새로운 문제를 만들기 위해 솔버 신규생산

d0 = diagnose_human.IntVar(
    0,
    diagnose_human.infinity(),
    "P0",
)
d1 = diagnose_human.IntVar(
    0,
    diagnose_human.infinity(),
    "P1",
)
d2 = diagnose_human.IntVar(
    0,
    diagnose_human.infinity(),
    "P2",
)
d3 = diagnose_human.IntVar(
    0,
    diagnose_human.infinity(),
    "P3",
)

# 결정변수는 동일하나 진단 검토용이므로 DO~D3을 새로 생성

diagnose_human.Add(
    6*d0 + 12*d1 + 14*d2 + 28*d3 <= 60
)

# 류지환 인시 조건은 그대로 유지.

diagnose_human.Add(
    6*d0 + 14*d1 + 18*d2 + 42*d3 <= 65
)
# Token 목표도 그대로 유지한다.

diagnose_human.Add(
    d3 >= 1
)
# P3 최소 한 건도 그대로 유지한다.

diagnose_human.Maximize(
    20*d0 + 30*d1 + 40*d2 + 100*d3
)
# 나머지 세 조건을 절대로 깨지 않을 시, 최대 얼마까지 벌 수 있을까?

diag_status = diagnose_human.Solve()

print(f"{diag_status}")

max_profit_under_other_goals = (
    diagnose_human.Objective().Value()
)

diagnose_result_human = pd.DataFrame([
    {
        "P0": d0.solution_value(),
        "P1": d1.solution_value(),
        "P2": d2.solution_value(),
        "P3": d3.solution_value(),
        "최대이익": max_profit_under_other_goals,
        "이익하한": 210,
        "갭": (
            210 - max_profit_under_other_goals
        ),

    }
])

display(diagnose_result_human)

print(f"왜 {max_profit_under_other_goals} 인가?")
print(f"Human≤60h, Token≤650,000, P3≥1을 유지할 경우 =, {max_profit_under_other_goals:.0f}만원이며")
print(f"원래 목표 210만원에 비해,  {210-max_profit_under_other_goals:.0f}만원 부족하다")



0


,P0,P1,P2,P3,최대이익,이익하한,갭
0,3.0,0.0,0.0,1.0,160.0,210,50.0


왜 160.0 인가?
Human≤60h, Token≤650,000, P3≥1을 유지할 경우 =, 160만원이며
원래 목표 210만원에 비해,  50만원 부족하다


# STEP 결론 : 이제 INFEASIBLE의 의미가 달라진다.

단순히

$$안 됨.$$

이 아니라,

$$다른 목표를 모두 지키면 이익은 최대 160만원$$

이라는 뜻이다.

따라서 다음 질문이 자연스럽다.

**210만원을 포기할까?**
**Human Time을 늘릴까?**
**Token을 더 쓸까?**

이게 *Goal Programming*이 필요한 이유다.